# Deduplicate Data

Remove near-similar tweet

## A. Overview

Apply Reciprocal Rank Fusion (RRF) Hybrid seach using n-gram and all-MiniLM-L6-v2 Transformer model to filter out similar tweet

In [ ]:
%pip install datasketch sentence-transformers
# install either faiss-cuda or faiss-cpu depending on your environment
# %pip install faiss-cuda
# %pip install faiss-cpu

## B. Combine Datasets

In [ ]:
from pathlib import Path
import csv
import os
import random
import json
import re

from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt

from collections import defaultdict

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from huggingface_hub import login

from datasketch import MinHash, MinHashLSH

import configuration
from src import data_utils, setup
from src.normalizer import similarity

login(os.getenv("HF_TOKEN"))

In [ ]:
dataset_path = Path('..') / 'data'

embedding_model="all-MiniLM-L6-v2"
nprobe = 10
chunk_size=50000
top_k = 100

### B.1. Load Datasets

In [ ]:
disaster_filepath = dataset_path / 'disaster'
df_disaster = pd.read_csv(disaster_filepath / 'full.csv')
df_disaster['subset'] = 'disaster'
df_disaster.shape

In [ ]:
# n_disaster = len(df_disaster)
# rule of thumb for nlist: 4 * sqrt(n) where n is the number of data points
nlist_disaster = int(4 * np.sqrt(len(df_disaster)))
embeddings_file_path = disaster_filepath / "disaster_informative_embeddings.npz"

similarity.filter_duplicates_faiss(
    df_disaster,
    nlist=nlist_disaster,
    chunk_size=chunk_size,
    embedding_model="all-MiniLM-L6-v2",
    train_sample_size=100000,
    nprobe=nprobe,
    search_type="radius",
    top_k=top_k,
    similarity_threshold=0.75,
    embeddings_file_path=embeddings_file_path,
    checkpoint_file="disaster_deduplication_checkpoint.json",
)

In [ ]:
# extended_filepath = dataset_path / "extended"

# df_weather = pd.read_csv(
#     extended_filepath / "weather.csv"
# )["tweet_text"].to_frame()
# df_weather["informative"] = False
# df_weather["subset"] = "weather"

# df_out_topic = pd.read_csv(
#     extended_filepath / "out_topic.csv"
# )["tweet_text"].to_frame()
# df_out_topic["informative"] = False
# df_out_topic["subset"] = "out_topic"

## C. Filtering

### C.1. Embedding

In [ ]:
embedding_model="all-MiniLM-L6-v2"
nprobe = 10
chunk_size=50000
similarity_threshold=0.6
top_k = 100

In [ ]:
# rule of thumb for nlist: 4 * sqrt(n) where n is the number of data points
n_disaster = len(df_disaster)
nlist_disaster = int(4 * np.sqrt(n_disaster))
train_sample_size_disaster = 80_000

df_disaster_embeddings = similarity.vectorize_faiss(
    df_disaster, text_column='tweet_text', model_name=embedding_model
)

In [ ]:
# save the embeddings to disk
np.savez(
    extended_filepath / "df_disaster_informative_embeddings.npz",
    embeddings=df_disaster_informative_embeddings,
    model_name=embedding_model,
)

In [ ]:
# # rule of thumb for nlist: 4 * sqrt(n) where n is the number of data points
# n_weather = len(df_weather)
# nlist_weather = int(4 * np.sqrt(n_weather))
# train_sample_size_weather = int(n_weather / 10)

# df_weather_embeddings = similarity.vectorize_faiss(
#     df_weather, text_column='tweet_text', model_name=embedding_model
# )

In [ ]:
# # save the embeddings to disk
# np.savez(
#     extended_filepath / "df_weather_embeddings.npz",
#     embeddings=df_weather_embeddings,
#     model_name=embedding_model,
# )

### C.2. Train

In [ ]:
train_sample_size_disaster_informative = 80_000
train_index_disaster_informative, gpu_index_disaster_informative = similarity.train_faiss_index(
    df_disaster_informative_embeddings,
    train_sample_size=train_sample_size_disaster_informative,
    nprobe=nprobe,
)

In [ ]:
# train_index_weather, gpu_index_weather = similarity.train_faiss_index(
#     df_weather_embeddings,
#     train_sample_size=train_sample_size_weather,
#     nprobe=nprobe,
# )

### C.3. Search

In [ ]:
# df_disaster_informative_drop_indices_radius = similarity.faiss_range_search(
#     df_disaster_informative_embeddings,
#     train_index_disaster_informative,
#     gpu_index=gpu_index_disaster_informative,
#     chunk_size=chunk_size,
#     similarity_threshold=similarity_threshold,
#     checkpoint_file='df_disaster_informative_drop_indices_radius.checkpoint.json',
# )

In [ ]:
# df_weather_drop_indeices_radius = similarity.faiss_range_search(
#     df_weather_embeddings,
#     train_index_weather,
#     gpu_index=gpu_index_weather,
#     chunk_size=chunk_size,
#     similarity_threshold=similarity_threshold,
#     checkpoint_file='df_weather_drop_indeices_radius.checkpoint.json',
# )

In [ ]:
# df_weather.drop(index=df_weather_drop_indeices_radius).reset_index(drop=True).to_csv(
#     extended_filepath / f"df_weather_radius_{similarity_threshold}.csv", index=False, quoting=csv.QUOTE_ALL)

In [ ]:
for similarity_threshold in [0.6, 0.65, 0.7, 0.75, 0.8]:
    df_disaster_informative_drop_indices_knearest = similarity.faiss_neighbors_search(
        df_disaster_informative_embeddings,
        train_index_disaster_informative,
        gpu_index=gpu_index_disaster_informative,
        chunk_size=chunk_size,
        top_k=top_k,
        similarity_threshold=similarity_threshold,
        checkpoint_file='df_disaster_informative_drop_indices_knearest.checkpoint.json',
    )
    df_disaster_informative.drop(
        index=df_disaster_informative_drop_indices_knearest
    ).reset_index(drop=True).to_csv(
        extended_filepath / f"df_disaster_informative_knearest_{similarity_threshold}_{top_k}.csv", index=False, quoting=csv.QUOTE_ALL)

In [ ]:
# df_weather_drop_indices_knearest = similarity.faiss_neighbors_search(
#     df_weather_embeddings,
#     train_index_weather,
#     gpu_index=gpu_index_weather,
#     chunk_size=chunk_size,
#     top_k=top_k,
#     similarity_threshold=similarity_threshold,
#     checkpoint_file='df_weather_knearest.checkpoint.json',
# )

In [ ]:
# df_weather.drop(index=df_weather_drop_indices_knearest).reset_index(drop=True).to_csv(
#     extended_filepath / f"df_weather_knearest_{top_k}.csv", index=False, quoting=csv.QUOTE_ALL)

In [ ]:
# rule of thumb for nlist: 4 * sqrt(n) where n is the number of data points
n_out_topic = len(df_out_topic)
nlist_out_topic = int(4 * np.sqrt(n_out_topic))
train_sample_size_out_topic = int(n_out_topic / 10)

In [ ]:
# df_out_topic_embeddings = similarity.vectorize_faiss(
#     df_out_topic, text_column='tweet_text', model_name=embedding_model, device='cuda'
# )
# np.savez(
#     extended_filepath / "df_out_topic_embeddings.npz",
#     embeddings=df_out_topic_embeddings,
#     model_name=embedding_model,
# )

df_out_topic_embeddings = np.load(
    extended_filepath / "df_out_topic_embeddings.npz",
    allow_pickle=True
)['embeddings']

In [ ]:
len(df_out_topic_embeddings)

In [ ]:
train_index_out_topic, gpu_index_out_topic = similarity.train_faiss_index(
    df_out_topic_embeddings,
    train_sample_size=train_sample_size_out_topic,
    nprobe=nprobe,
)

In [ ]:
df_out_topic_drop_indeices_radius = similarity.faiss_range_search(
    df_out_topic_embeddings,
    train_index_out_topic,
    gpu_index=gpu_index_out_topic,
    chunk_size=chunk_size,
    similarity_threshold=similarity_threshold,
    checkpoint_file='df_out_topic_drop_indeices_radius.checkpoint.json',
)

In [ ]:
df_out_topic.drop(index=df_out_topic_drop_indeices_radius).reset_index(drop=True).to_csv(
    extended_filepath / f"df_out_topic_radius_{similarity_threshold}.csv", index=False, quoting=csv.QUOTE_ALL)

In [ ]:
df_out_topic_knearest = similarity.faiss_neighbors_search(
    df_out_topic_embeddings,
    train_index_out_topic,
    gpu_index=gpu_index_out_topic,
    chunk_size=chunk_size,
    top_k=top_k,
    similarity_threshold=similarity_threshold,
    checkpoint_file='df_out_topic_knearest.checkpoint.json',
)

In [ ]:
df_out_topic.drop(index=df_out_topic_knearest).reset_index(drop=True).to_csv(
    extended_filepath / f"df_out_topic_knearest_{similarity_threshold}_top_{top_k}.csv", index=False, quoting=csv.QUOTE_ALL)